# Assignment 6 — CNN for Tomato Leaf Disease Classification

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** GPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and dataset

Classify tomato leaf images by disease using a custom Convolutional
Neural Network. We use the tomato classes from TensorFlow Datasets'
PlantVillage collection. The pipeline filters and remaps the original
labels, resizes images, augments only training data, and evaluates on a
separate test split.

The first download is large. Use a GPU runtime and Colab's standard disk.
`MAX_TRAIN_IMAGES` can be lowered for a short demonstration.


In [ ]:
# %pip install -q -U tensorflow-datasets
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = 128
BATCH_SIZE = 32
MAX_TRAIN_IMAGES = 5000

builder = tfds.builder("plant_village")
builder.download_and_prepare()
label_names = builder.info.features["label"].names
tomato_ids = [i for i, name in enumerate(label_names) if name.startswith("Tomato")]
tomato_names = [label_names[i].replace("Tomato___", "") for i in tomato_ids]
print(dict(enumerate(tomato_names)))


In [ ]:
raw_train = builder.as_dataset(split="train[:70%]", as_supervised=True)
raw_val = builder.as_dataset(split="train[70%:85%]", as_supervised=True)
raw_test = builder.as_dataset(split="train[85%:]", as_supervised=True)

keys = tf.constant(tomato_ids, dtype=tf.int64)
values = tf.range(len(tomato_ids), dtype=tf.int64)
lookup = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(keys, values), default_value=-1
)

def is_tomato(image, label):
    return lookup.lookup(label) >= 0

def prepare(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return tf.cast(image, tf.float32) / 255.0, lookup.lookup(label)

def pipeline(ds, training=False, limit=None):
    ds = ds.filter(is_tomato).map(prepare, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2000, seed=SEED)
    if limit:
        ds = ds.take(limit)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = pipeline(raw_train, training=True, limit=MAX_TRAIN_IMAGES)
val_ds = pipeline(raw_val)
test_ds = pipeline(raw_test)

images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(image); ax.set_title(tomato_names[int(label)]); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
], name="augmentation")

model = tf.keras.Sequential([
    tf.keras.layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    augmentation,
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(len(tomato_names), activation="softmax"),
], name="tomato_cnn")
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )],
)


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
print(f"Test accuracy: {test_accuracy:.3f}")
print(classification_report(y_true, y_pred, target_names=tomato_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=tomato_names, yticklabels=tomato_names)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Tomato disease confusion matrix")
plt.xticks(rotation=60, ha="right"); plt.tight_layout(); plt.show()


## Discussion

Convolution layers learn local visual patterns such as spots, colour
changes, and leaf texture. Augmentation reduces sensitivity to rotation
and framing. Class-level recall is especially important for diseases that
should not be missed. PlantVillage has clean backgrounds, so performance
may decrease on real field photographs.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
